# 공통 설정: LLM 제공자 선택

원본: [`llm.py`](llm.py)

`LLM_PROVIDER`에 따라 사내 게이트웨이 또는 OpenAI 클라이언트를 생성한다. 강의 예제는 이 함수를 불러와 같은 호출 방식을 사용한다.

## 이번 실습 목표

환경변수 → 제공자 분기 → 필수값 검사 → 클라이언트 생성 흐름을 추적한다. 시크릿 값은 출력하지 않는다.

## 실행 전 확인

원본 정의를 읽고 실행할 수 있지만, 실제 `get_llm()` 호출은 사내망 또는 사외 API 설정과 자격증명이 준비된 경우에만 한다. 설정값이나 헤더를 출력하지 않는다.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()


def _required(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f'{name} 환경변수가 필요합니다. 실행할 환경 파일을 확인하세요.')
    return value

def get_llm(**kwargs):
    """
    LLM_PROVIDER 환경변수에 따라 LLM 인스턴스를 반환한다.
    - openai: OpenAI ChatOpenAI (사외)
    - samsung (기본값): 삼성 사내 LLM
    """
    provider = os.getenv("LLM_PROVIDER", "samsung").strip().lower()

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        if "model" not in kwargs:
            kwargs["model"] = _required("OPENAI_MODEL")
        if kwargs["model"] == "gpt-6-luna":
            # 이 수업의 도구 호출 실습은 Chat Completions를 사용한다.
            kwargs.setdefault("reasoning_effort", "none")
            # gpt-6 계열은 도구가 있으면 Responses API로 자동 전환되므로 고정한다.
            kwargs.setdefault("use_responses_api", False)
        return ChatOpenAI(api_key=_required("OPENAI_API_KEY"), **kwargs)

    elif provider == "samsung":
        from langchain_openai import ChatOpenAI
        from config.samsung import llm_config
        samsung = llm_config()
        kwargs.setdefault("model", samsung["model"])
        return ChatOpenAI(
            # 사내 게이트웨이는 x-dep-ticket으로 인증한다. SDK 필수값만 채운다.
            api_key="samsung-gateway",
            base_url=samsung["base_url"],
            default_headers=samsung["headers"],
            extra_body={
                "chat_template_kwargs": {
                    "enable_thinking": False,
                }
            },
            **kwargs,
        )

    raise ValueError(f"지원하지 않는 LLM_PROVIDER: {provider!r}. 'openai' 또는 'samsung'을 사용하세요.")


## CodeMate와 함께하기

CodeMate에게 `get_llm`의 분기와 필수 환경변수를 표로 정리하게 한다. 잘못된 제공자 이름과 누락된 값에서 어떤 오류가 나는지도 코드 근거로 검토한다.

## 결과 검토

사내 경로의 인증 헤더는 어떤 이름을 사용하는가? `LLM_PROVIDER`를 잘못 적으면 어떤 예외가 나는가? 사외 실행에 필요한 값을 코드에서 찾아보되 값 자체는 적지 않는다.